In [1]:
import pandas as pd
import numpy as np

Import request

## Step 1: Mapping SNOMED CT Codes to ICD-10-CM

To evaluate whether a rural triage unit can maintain financial solvency, clinical
condition records must first be translated into billable diagnostic codes. This step
uses the UMLS REST API crosswalk endpoint (accessed August 2026) to map SNOMED CT
condition codes to ICD-10-CM.

**Method note.** This uses the UMLS `/crosswalk` endpoint, which returns codes sharing
a UMLS concept identifier — not the rule-based NLM SNOMED CT to ICD-10-CM Map. NLM
states that this synonymy has not been rigorously tested for clinical use and that
results should be reviewed for relevancy. The crosswalk carries no map groups,
priorities, or age/sex conditional rules. It was chosen because it requires only an API
key rather than a manually uploaded licensed file, allowing the notebook to run from a
cold start.

Where multiple ICD-10-CM codes are returned, the most specific (longest) is selected,
since ICD-10-CM billing requires the most specific code available — `E11` is a category,
`E11.9` is billable.

Performed strictly as a sustainability demonstration on synthetic data, not as a clinical
revenue tool.

In [11]:
import requests
from google.colab import userdata

UMLS_KEY = userdata.get('UMLS_KEY')
UTS_BASE = "https://uts-ws.nlm.nih.gov/rest"

def snomed_to_icd10(snomed_code):
    """
    Crosswalk a SNOMED CT code to ICD-10-CM via UMLS CUI synonymy.

    Returns (best_code, all_codes, status). The 'best' code is the longest
    returned, i.e. the most specific — ICD-10-CM billing requires the most
    specific code available, so a 3-character category like 'E11' is not
    billable while 'E11.9' is.

    NOTE: This is the UMLS /crosswalk endpoint, NOT the NLM SNOMED CT to
    ICD-10-CM Map. It returns codes sharing a UMLS concept identifier, without
    map rules, groups, or priorities. NLM states this synonymy has not been
    rigorously tested for clinical use. Documented limitation.
    """
    url = f"{UTS_BASE}/crosswalk/current/source/SNOMEDCT_US/{snomed_code}"
    r = requests.get(url, params={"targetSource": "ICD10CM", "apiKey": UMLS_KEY},
                     timeout=30)
    if r.status_code == 404:
        return None, [], "No ICD-10-CM crosswalk found"
    r.raise_for_status()
    results = r.json().get('result', [])
    codes = [x['ui'] for x in results if x.get('ui')]
    if not codes:
        return None, [], "Empty result"
    best = max(codes, key=len)          # most specific
    return best, codes, "OK"


for code, label in [('44054006', 'Type 2 diabetes'), ('195967001', 'Asthma')]:
    best, codes, status = snomed_to_icd10(code)
    print(f"{label:<20} SNOMED {code} -> best={best}  all={codes}  [{status}]")

Type 2 diabetes      SNOMED 44054006 -> best=E11  all=['E11']  [OK]
Asthma               SNOMED 195967001 -> best=J45.909  all=['J45', 'J45.909', 'J45.90']  [OK]


### Step 2: Validating ICD-10-CM Codes Against Official Standards
Before any mapped diagnostic codes are passed further down the reimbursement chain, they must be verified to prevent invalid values from propagating silently. This block parses the official fixed-width text files from the [CDC/NCHS ICD-10-CM 2027 Set](https://ftp.cdc.gov/pub/Health_Statistics/NCHS/Publications/ICD10CM/2027/) (accessed August 2026) to confirm that every generated code actively exists in the official federal dataset. Unmapped or unrecognized codes are explicitly tracked rather than silently dropped to ensure data transparency.

In [17]:
import requests, zipfile, io

CDC_URL = ("https://ftp.cdc.gov/pub/Health_Statistics/NCHS/Publications/"
           "ICD10CM/2027/icd10cm-code-descriptions-2027.zip")

r = requests.get(CDC_URL, timeout=120)
r.raise_for_status()
z = zipfile.ZipFile(io.BytesIO(r.content))
print("Files in archive:", z.namelist())



Files in archive: ['icd10cm-code-descriptions-2027/', 'icd10cm-code-descriptions-2027/icd10cm-codes-2027.txt', 'icd10cm-code-descriptions-2027/icd10cm-codes-addenda-2027.txt', 'icd10cm-code-descriptions-2027/icd10cm-order-2027.txt', 'icd10cm-code-descriptions-2027/icd10cm-order-addenda-2027.txt', 'icd10cm-code-descriptions-2027/icd10cmCodesFile.pdf', 'icd10cm-code-descriptions-2027/icd10OrderFiles.pdf']


In [16]:
import pandas as pd

path = [n for n in z.namelist() if n.endswith('icd10cm-order-2027.txt')][0]
with z.open(path) as f:
    cdc = pd.read_fwf(f, colspecs=[(0,5),(6,13),(14,15),(16,76),(77,None)],
                      header=None, names=['order','code','billable','short','long'],
                      dtype={'code':str,'billable':str}, encoding='latin-1')

cdc['code'] = cdc['code'].str.strip()
valid_icd10 = set(cdc['code'])
billable_icd10 = set(cdc.loc[cdc['billable']=='1', 'code'])
print(f"{len(cdc):,} ICD-10-CM codes | {len(billable_icd10):,} billable")

def validate_icd10(code):
    """Returns (exists, is_billable). CDC stores codes undotted."""
    if code is None:
        return False, False
    clean = str(code).replace('.', '').strip().upper()
    return clean in valid_icd10, clean in billable_icd10

for c in ['E11', 'E11.9', 'J45.909', 'J45', 'ZZZZZ']:
    ex, bl = validate_icd10(c)
    print(f"  {c:<10} exists={ex}  billable={bl}")

98,403 ICD-10-CM codes | 74,879 billable
  E11        exists=True  billable=False
  E11.9      exists=True  billable=True
  J45.909    exists=True  billable=True
  J45        exists=True  billable=False
  ZZZZZ      exists=False  billable=False


## DRG Approximation & Payment Weight Estimate

### Step 3 & 4: Approximating DRGs and Estimating Payment Weight
*Methodological Limitation:* Inpatient Diagnosis-Related Groups (DRGs) are officially assigned per complete hospital stay—considering principal diagnoses, secondary complications, and procedures—rather than a single point-of-care diagnosis. Because this tool acts as a rapid rural triage approximation, we use the patient's primary condition as a practical diagnostic proxy matched via the [NBER DRG–MDC Crosswalk](https://www.nber.org/research/data/diagnosis-related-group-major-diagnostic-category-crosswalk) (accessed August 2026).

Using the retrieved relative weight multiplied against the documented CMS Base Rate (FY2024 IPPS standardized amount of $6,497.77, sourced from [CMS.gov](https://www.cms.gov)), this code calculates an estimated encounter value to evidence financial viability.

In [19]:
import requests, zipfile, io, re, glob
from collections import defaultdict

# --- Download the CMS MS-DRG Definitions Manual (V44, FY2027) ---
MSDRG_URL = "https://www.cms.gov/files/zip/fy2027-fr-icd10-ms-drg-definitions-manual-files-v44.zip"
r = requests.get(MSDRG_URL, timeout=180); r.raise_for_status()
zdrg = zipfile.ZipFile(io.BytesIO(r.content))
mdc_files = [n for n in zdrg.namelist() if 'mdcs_' in n.lower() and n.endswith('.txt')]
print(f"MDC files: {mdc_files}")

# --- Parse ICD-10 -> MDC ---
MDC_HDR = re.compile(r'^MDC (\d{2}) Assignment of Diagnosis Codes\s*$')
CODE_LN = re.compile(r'^\s{2}([A-Z][A-Z0-9]{2,7})\s{2,}(.+?)\s*$')

icd_to_mdc = {}
for name in mdc_files:
    in_block = False; cur = None
    for raw in zdrg.open(name):
        line = raw.decode('latin-1').rstrip('\r\n')
        m = MDC_HDR.match(line)
        if m:
            cur, in_block = m.group(1), True; continue
        if line.strip() and not line.startswith('  ') and in_block:
            in_block = False; continue
        m = CODE_LN.match(line)
        if m and in_block and cur:
            icd_to_mdc.setdefault(m.group(1), cur)

print(f"{len(icd_to_mdc):,} ICD-10 codes mapped to MDC")

# --- MDC -> DRG weight range (NBER, already loaded as nber_df) ---
nber_df['mdc'] = nber_df['mdc'].astype(str).str.strip().str.zfill(2)
nber_df['weights'] = pd.to_numeric(nber_df['weights'], errors='coerce')
mdc_weights = (nber_df.dropna(subset=['weights'])
                      .groupby('mdc')['weights']
                      .agg(['min','max','mean','count']).round(4))

def icd10_to_mdc_weights(icd10_code):
    """ICD-10 -> MDC -> DRG weight range for that MDC."""
    clean = str(icd10_code).replace('.', '').strip().upper()
    mdc = icd_to_mdc.get(clean)
    if mdc is None or mdc not in mdc_weights.index:
        return None
    row = mdc_weights.loc[mdc]
    return {'mdc': mdc, 'weight_min': row['min'], 'weight_max': row['max'],
            'weight_mean': row['mean'], 'n_drgs': int(row['count'])}

for c in ['E11.9', 'J45.909', 'I10']:
    print(f"  {c:<10} {icd10_to_mdc_weights(c)}")

MDC files: ['mdcs_00_07.txt', 'mdcs_08_11.txt', 'mdcs_12_21.txt', 'mdcs_22_25.txt']
22,436 ICD-10 codes mapped to MDC
  E11.9      {'mdc': '10', 'weight_min': np.float64(0.6212), 'weight_max': np.float64(3.7268), 'weight_mean': np.float64(1.7735), 'n_drgs': 26}
  J45.909    {'mdc': '04', 'weight_min': np.float64(0.6285), 'weight_max': np.float64(6.4347), 'weight_mean': np.float64(1.5272), 'n_drgs': 41}
  I10        {'mdc': '05', 'weight_min': np.float64(0.4551), 'weight_max': np.float64(11.3188), 'weight_mean': np.float64(3.2464), 'n_drgs': 101}


CMS BASE RATE

**Limitation: category-level crosswalk results drop out of the estimate.**

The UMLS crosswalk sometimes returns only a 3-character ICD-10 category rather
than a billable code — type 2 diabetes (SNOMED 44054006) returns `E11`, not
`E11.9`. Such codes fail two downstream checks: they are not billable per the
CDC 2027 order file, and they carry no MDC assignment in the MS-DRG Definitions
Manual, which assigns MDCs at billable-code level only. A condition that
crosswalks to a category therefore contributes nothing to the weight estimate.

This is a consequence of using the UMLS CUI-synonymy crosswalk rather than the
rule-based NLM SNOMED CT to ICD-10-CM Map, which returns billable targets under
its map rules. Estimates are therefore **conservative** — undercounting rather
than overstating encounter value. Mapping coverage is reported below.

In [20]:
CMS_BASE_RATE = 6497.77          # FY2024 IPPS standardized amount — verify vintage

def estimate_encounter_value(snomed_codes, age, sex):
    """
    SNOMED conditions -> ICD-10-CM -> MDC -> DRG weight range.
    Sustainability demonstration on synthetic data. Not a payment prediction.
    """
    icd10, unmapped, mdcs = [], [], []

    for code in snomed_codes:
        best, all_codes, status = snomed_to_icd10(code)
        if best is None:
            unmapped.append({'snomed': code, 'reason': status}); continue
        exists, billable = validate_icd10(best)
        if not exists:
            unmapped.append({'snomed': code, 'reason': f'{best} not in CDC 2027'}); continue
        icd10.append({'code': best, 'billable': billable})
        w = icd10_to_mdc_weights(best)
        if w:
            mdcs.append({**w, 'icd10': best})

    primary = max(mdcs, key=lambda x: x['weight_mean']) if mdcs else None

    return {
        'icd10_codes':  [c['code'] for c in icd10],
        'billable_codes': [c['code'] for c in icd10 if c['billable']],
        'unmapped':     unmapped,
        'mdc':          primary['mdc'] if primary else None,
        'weight_range': (primary['weight_min'], primary['weight_max']) if primary else None,
        'weight_mean':  primary['weight_mean'] if primary else None,
        'estimate_usd': round(primary['weight_mean'] * CMS_BASE_RATE, 2) if primary else None,
        'caveats': [
            'NON-CLINICAL USE ONLY. Synthetic Synthea data.',
            'MDC-level approximation. CMS states MS-DRG assignment requires principal '
            'diagnosis, up to 24 secondary diagnoses, up to 25 procedures, and in some '
            'cases age, sex and discharge status — inputs unavailable at triage.',
            'Weight reflects clinical resource intensity, not payment. Critical Access '
            'and Rural Emergency Hospitals are cost-reimbursed, not DRG-paid.',
            'SNOMED-to-ICD-10 via UMLS CUI crosswalk, not the rule-based NLM Map.',
        ],
    }

'Conditions crosswalking only to a 3-character ICD-10 category (e.g. E11 for '
'type 2 diabetes) contribute nothing to the estimate — such codes are not '
'billable and carry no MDC assignment. Estimates are conservative.',

In [21]:
result = estimate_encounter_value(['44054006', '195967001'], age=84, sex='f')

for k, v in result.items():
    if k == 'caveats':
        print(f"\ncaveats:")
        for c in v: print(f"  - {c}")
    else:
        print(f"{k}: {v}")

icd10_codes: ['E11', 'J45.909']
billable_codes: ['J45.909']
unmapped: []
mdc: 04
weight_range: (np.float64(0.6285), np.float64(6.4347))
weight_mean: 1.5272
estimate_usd: 9923.39

caveats:
  - NON-CLINICAL USE ONLY. Synthetic Synthea data.
  - MDC-level approximation. CMS states MS-DRG assignment requires principal diagnosis, up to 24 secondary diagnoses, up to 25 procedures, and in some cases age, sex and discharge status — inputs unavailable at triage.
  - Weight reflects clinical resource intensity, not payment. Critical Access and Rural Emergency Hospitals are cost-reimbursed, not DRG-paid.
  - SNOMED-to-ICD-10 via UMLS CUI crosswalk, not the rule-based NLM Map.


In [22]:
for c in ['E11', 'J45909']:
    print(c, icd10_to_mdc_weights(c))

E11 None
J45909 {'mdc': '04', 'weight_min': np.float64(0.6285), 'weight_max': np.float64(6.4347), 'weight_mean': np.float64(1.5272), 'n_drgs': 41}


### Loading file to inspect Columns

In [5]:
# Load the file to inspect columns
NBER_URL = "https://data.nber.org/drg/csv/drgweight2026FR.csv"
nber_df = pd.read_csv(NBER_URL)
print(f"Loaded {len(nber_df)} DRGs from NBER (FY2026 Final Rule)")
print(nber_df.columns.tolist())

Loaded 773 DRGs from NBER (FY2026 Final Rule)
['ms_drg', 'pa_drg', 'nprm_drg', 'mdc', 'type', 'msdrg_title', 'weights', 'los_geo', 'los_mean']


### Crosswalk Logic

Methodological Limitation: In this notebook, I approximate the DRG using the patient's primary condition as a proxy. Real DRG assignment requires a comprehensive "grouper" that considers the entire inpatient stay (procedures, comorbidities, discharge status). This tool is strictly a sustainability approximation, not an exact revenue or grouping assignment.

## The Interface Function

#### Wrap everything into a strict dictionary signature

### The Final Interface

### Step 5: Constructing the Final Encounter Value Interface
To seamlessly integrate this evaluation pipeline into the existing Streamlit demonstration app without requiring major structural refactoring, all previous mapping, validation, and weighting logic is wrapped into a single Python function: `estimate_encounter_value(snomed_codes, age, sex)`. This function processes a patient cohort, tracks structural unmapped gaps, and returns a strictly formatted dictionary containing the estimated reimbursement figures alongside mandatory non-clinical caveats.

In [6]:
import json

def format_encounter_result(result_dict):
    """
    Takes the dictionary from estimate_encounter_value and prints it
    in a clean, human-readable, structured format.
    """
    print("=" * 60)
    print("           PATIENT ENCOUNTER REIMBURSEMENT ESTIMATE          ")
    print("=" * 60)

    # Validated Codes
    print("\n[+] Mapped & Validated ICD-10 Codes:")
    if result_dict['icd10_codes']:
        for code in result_dict['icd10_codes']:
            print(f"    • {code}")
    else:
        print("    (None)")

    # Unmapped Codes
    print("\n[-] Unmapped / Failed Codes:")
    if result_dict['unmapped']:
        for item in result_dict['unmapped']:
            print(f"    • SNOMED: {item['snomed']} | Reason: {item['reason']}")
    else:
        print("    (None)")

    # Financial & Grouping Summary
    print("\n[#] Reimbursement & Grouping Summary:")
    print(f"    • Approximated DRG : {result_dict['drg'] or 'N/A'}")
    print(f"    • Major Diagnostic Category (MDC): {result_dict['mdc'] or 'N/A'}")
    print(f"    • DRG Relative Weight: {result_dict['weight'] or 'N/A'}")

    est_usd = result_dict['estimate_usd']
    formatted_usd = f"${est_usd:,.2f}" if est_usd is not None else "N/A"
    print(f"    • Estimated Value    : {formatted_usd}")

    # Mandatory Caveats
    print("\n[!] Methodological Caveats:")
    for caveat in result_dict['caveats']:
        print(f"    * {caveat}")
    print("=" * 60)

# Example usage with your dictionary output:
sample_output = {
    'icd10_codes': [],
    'unmapped': [
        {'snomed': '44054006', 'reason': 'Failed CDC Validation'},
        {'snomed': '195967001', 'reason': 'Failed CDC Validation'}
    ],
    'drg': None,
    'mdc': None,
    'weight': None,
    'estimate_usd': None,
    'caveats': [
        'NON-CLINICAL USE ONLY. Data is synthetic.',
        'DRG approximated from primary diagnosis proxy. Actual grouping requires full inpatient context.'
    ]
}

format_encounter_result(sample_output)

           PATIENT ENCOUNTER REIMBURSEMENT ESTIMATE          

[+] Mapped & Validated ICD-10 Codes:
    (None)

[-] Unmapped / Failed Codes:
    • SNOMED: 44054006 | Reason: Failed CDC Validation
    • SNOMED: 195967001 | Reason: Failed CDC Validation

[#] Reimbursement & Grouping Summary:
    • Approximated DRG : N/A
    • Major Diagnostic Category (MDC): N/A
    • DRG Relative Weight: N/A
    • Estimated Value    : N/A

[!] Methodological Caveats:
    * NON-CLINICAL USE ONLY. Data is synthetic.
    * DRG approximated from primary diagnosis proxy. Actual grouping requires full inpatient context.


Checking the mapping, the decimals places for the first 20 results.

# Modify the Condition extraction to keep codes alongside display names

In [7]:
# Modify the Condition extraction to keep codes alongside display names
record['condition_codes'] = [
    c['code']['coding'][0]['code']
    for c in resources if c['resourceType'] == 'Condition'
    and c.get('code', {}).get('coding')
]

# Then in export_cohort.py, add:
app_data[case]['reimbursement'] = estimate_encounter_value(
    r['condition_codes'], r['age'], r['gender']
)

NameError: name 'resources' is not defined

Add streamlit application to this code

In [ ]:
st.subheader("Encounter Value Estimate")
st.caption("Demonstration only — synthetic data, approximated DRG assignment.")

rb = p.get('reimbursement')
if rb:
    c1, c2, c3 = st.columns(3)
    c1.metric("DRG", rb['drg'] or "—")
    c2.metric("Relative weight", f"{rb['weight']:.4f}" if rb['weight'] else "—")
    c3.metric("Estimated value",
              f"${rb['estimate_usd']:,.0f}" if rb['estimate_usd'] else "—")

    if rb['icd10_codes']:
        st.write("**Mapped ICD-10:** " + ", ".join(rb['icd10_codes']))
    if rb['unmapped']:
        with st.expander(f"{len(rb['unmapped'])} unmapped code(s)"):
            for u in rb['unmapped']:
                st.write(f"• SNOMED {u['snomed']} — {u['reason']}")
    for c in rb['caveats']:
        st.caption(f"⚠ {c}")
else:
    st.info("No reimbursement estimate available for this patient.")

In [ ]:
# Final cell of cloud_based_diagnostic_tool.ipynb
COHORT_SNOMED = {
    'Pediatric':          {'codes': [...], 'age': 0,   'sex': 'm'},
    'Healthy adult':      {'codes': [...], 'age': 39,  'sex': 'm'},
    'Complex geriatric':  {'codes': [...], 'age': 84,  'sex': 'f'},
    'Allergy carrier':    {'codes': [...], 'age': 17,  'sex': 'f'},
    'High medication':    {'codes': [...], 'age': 102, 'sex': 'm'},
}

estimates = {case: estimate_encounter_value(v['codes'], v['age'], v['sex'])
             for case, v in COHORT_SNOMED.items()}

import json
with open('reimbursement_estimates.json', 'w') as f:
    json.dump(estimates, f, indent=2)